# Quora Question Pairs - Duplicate Detection (RoBERTa cross-encoder, GPU)
Dataset: `questions.csv` (columns: `id, qid1, qid2, question1, question2, is_duplicate`).
Since there is no labelled test set, the data is split **80% train / 10% validation / 10% test**.
- Validation set: model selection + F1-optimal threshold search
- Test set: final, untouched F1 score

**Settings:** Accelerator = GPU (T4 x2 / P100), Internet = ON.

In [6]:
import os, glob, time, random
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, log_loss, classification_report, accuracy_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

CFG = dict(
    model_name="roberta-base",   # stronger: "microsoft/deberta-v3-base" (slower) | faster: "distilroberta-base"
    max_len=64,
    epochs=3,
    lr=2e-5,
    train_bs=64,
    eval_bs=512,
    warmup=0.06,
    wd=0.01,
    seed=42,
    tta=True,                    # average predictions over both question orders
)

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(CFG["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| GPUs:", torch.cuda.device_count())

Device: cuda | GPUs: 2


In [7]:
# ---------------- Load data ----------------
path = sorted(glob.glob("/kaggle/input/**/questions.csv", recursive=True))[0]
print("Loading:", path)
df = pd.read_csv(path)
df = df.dropna(subset=["question1", "question2"]).reset_index(drop=True)   # drops the 2 rows with missing text
df["question1"] = df["question1"].astype(str)
df["question2"] = df["question2"].astype(str)
print(df.shape, "| duplicate rate:", round(df.is_duplicate.mean(), 3))

# ---------------- 80 / 10 / 10 split ----------------
tr_df, tmp_df = train_test_split(df, test_size=0.2, stratify=df.is_duplicate, random_state=CFG["seed"])
va_df, te_df  = train_test_split(tmp_df, test_size=0.5, stratify=tmp_df.is_duplicate, random_state=CFG["seed"])
tr_df, va_df, te_df = [d.reset_index(drop=True) for d in (tr_df, va_df, te_df)]
print("train/val/test:", len(tr_df), len(va_df), len(te_df))

Loading: /kaggle/input/datasets/quora/question-pairs-dataset/questions.csv
(404348, 6) | duplicate rate: 0.369
train/val/test: 323478 40435 40435


In [8]:
# ---------------- Tokenizer, dataset, model ----------------
tok = AutoTokenizer.from_pretrained(CFG["model_name"])

class PairDS(torch.utils.data.Dataset):
    def __init__(self, d):
        self.q1, self.q2, self.y = d.question1.tolist(), d.question2.tolist(), d.is_duplicate.values
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.q1[i], self.q2[i], self.y[i]

def train_collate(batch):
    q1, q2, y = zip(*batch)
    swap = [random.random() < 0.5 for _ in q1]          # random order swap (augmentation)
    a = [b_ if s else a_ for a_, b_, s in zip(q1, q2, swap)]
    b = [a_ if s else b_ for a_, b_, s in zip(q1, q2, swap)]
    enc = tok(a, b, truncation=True, max_length=CFG["max_len"], padding=True, return_tensors="pt")
    enc["labels"] = torch.tensor(y, dtype=torch.long)
    return enc

train_dl = DataLoader(PairDS(tr_df), batch_size=CFG["train_bs"], shuffle=True,
                      collate_fn=train_collate, num_workers=2, drop_last=True)

model = AutoModelForSequenceClassification.from_pretrained(CFG["model_name"], num_labels=2)
model = model.float().to(device)   # force FP32 master weights (some checkpoints load as FP16, which breaks GradScaler)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
unwrap = lambda m: m.module if hasattr(m, "module") else m

no_decay = ["bias", "LayerNorm.weight"]
groups = [
    {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": CFG["wd"]},
    {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0},
]
opt = torch.optim.AdamW(groups, lr=CFG["lr"])
total_steps = len(train_dl) * CFG["epochs"]
sched = get_linear_schedule_with_warmup(opt, int(CFG["warmup"] * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
# ---------------- Inference + threshold helpers ----------------
@torch.no_grad()
def predict(q1, q2, tta=True):
    """Return P(duplicate) for lists of questions."""
    model.eval()
    n = len(q1)
    order = np.argsort([len(a) + len(b) for a, b in zip(q1, q2)])   # length-sorted batches = faster

    def run(A, B):
        res = np.zeros(n, dtype=np.float32)
        for i in range(0, n, CFG["eval_bs"]):
            idx = order[i:i + CFG["eval_bs"]]
            enc = tok([A[j] for j in idx], [B[j] for j in idx], truncation=True,
                      max_length=CFG["max_len"], padding=True, return_tensors="pt").to(device)
            with torch.amp.autocast("cuda"):
                logits = model(**enc).logits
            res[idx] = torch.softmax(logits.float(), -1)[:, 1].cpu().numpy()
        return res

    p = run(q1, q2)
    return (p + run(q2, q1)) / 2 if tta else p

def best_f1_threshold(y, p):
    best_t, best_f = 0.5, 0.0
    for t in np.arange(0.05, 0.95, 0.005):
        f = f1_score(y, (p > t).astype(int))
        if f > best_f: best_t, best_f = t, f
    return best_t, best_f

In [10]:
# ---------------- Training ----------------
best_f1, best_thr, best_state = -1, 0.5, None
for epoch in range(CFG["epochs"]):
    model.train()
    t0, run_loss = time.time(), 0.0
    for step, batch in enumerate(train_dl, 1):
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch.pop("labels")
        with torch.amp.autocast("cuda"):
            logits = model(**batch).logits
        loss = nn.functional.cross_entropy(logits.float(), labels)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        run_loss += loss.item()
        if step % 500 == 0:
            print(f"epoch {epoch+1} | step {step}/{len(train_dl)} | loss {run_loss/500:.4f} | {time.time()-t0:.0f}s")
            run_loss = 0.0

    val_p = predict(va_df.question1.tolist(), va_df.question2.tolist(), tta=CFG["tta"])
    thr, f1 = best_f1_threshold(va_df.is_duplicate.values, val_p)
    ll = log_loss(va_df.is_duplicate.values, np.clip(val_p, 1e-6, 1 - 1e-6))
    print(f"==== Epoch {epoch+1}: VAL F1 = {f1:.4f} @ threshold {thr:.3f} | logloss = {ll:.4f}")
    if f1 > best_f1:
        best_f1, best_thr = f1, thr
        best_state = {k: v.detach().cpu().clone() for k, v in unwrap(model).state_dict().items()}

unwrap(model).load_state_dict(best_state)
torch.save(best_state, "best_model.pt")
print(f"Best val F1 = {best_f1:.4f} (threshold {best_thr:.3f}) - weights saved to best_model.pt")

epoch 1 | step 500/5054 | loss 0.4966 | 116s
epoch 1 | step 1000/5054 | loss 0.3551 | 231s
epoch 1 | step 1500/5054 | loss 0.3272 | 346s
epoch 1 | step 2000/5054 | loss 0.3054 | 460s
epoch 1 | step 2500/5054 | loss 0.2985 | 575s
epoch 1 | step 3000/5054 | loss 0.2876 | 690s
epoch 1 | step 3500/5054 | loss 0.2809 | 805s
epoch 1 | step 4000/5054 | loss 0.2722 | 919s
epoch 1 | step 4500/5054 | loss 0.2692 | 1034s
epoch 1 | step 5000/5054 | loss 0.2622 | 1149s
==== Epoch 1: VAL F1 = 0.8660 @ threshold 0.465 | logloss = 0.2450
epoch 2 | step 500/5054 | loss 0.2266 | 115s
epoch 2 | step 1000/5054 | loss 0.2268 | 230s
epoch 2 | step 1500/5054 | loss 0.2191 | 345s
epoch 2 | step 2000/5054 | loss 0.2187 | 459s
epoch 2 | step 2500/5054 | loss 0.2210 | 574s
epoch 2 | step 3000/5054 | loss 0.2161 | 689s
epoch 2 | step 3500/5054 | loss 0.2130 | 804s
epoch 2 | step 4000/5054 | loss 0.2115 | 918s
epoch 2 | step 4500/5054 | loss 0.2151 | 1033s
epoch 2 | step 5000/5054 | loss 0.2100 | 1148s
==== Epoch 

In [11]:
# ---------------- Final evaluation on the held-out TEST split ----------------
te_p = predict(te_df.question1.tolist(), te_df.question2.tolist(), tta=CFG["tta"])
te_pred = (te_p > best_thr).astype(int)
y = te_df.is_duplicate.values

print(f"TEST F1       : {f1_score(y, te_pred):.4f}   (threshold {best_thr:.3f} chosen on validation)")
print(f"TEST accuracy : {accuracy_score(y, te_pred):.4f}")
print(f"TEST logloss  : {log_loss(y, np.clip(te_p, 1e-6, 1-1e-6)):.4f}")
print()
print(classification_report(y, te_pred, target_names=["not duplicate", "duplicate"], digits=4))

out = te_df[["id", "question1", "question2", "is_duplicate"]].copy()
out["pred_prob"], out["pred_label"] = te_p, te_pred
out.to_csv("test_predictions.csv", index=False)

TEST F1       : 0.8806   (threshold 0.425 chosen on validation)
TEST accuracy : 0.9081
TEST logloss  : 0.2357

               precision    recall  f1-score   support

not duplicate     0.9497    0.9021    0.9253     25505
    duplicate     0.8459    0.9184    0.8806     14930

     accuracy                         0.9081     40435
    macro avg     0.8978    0.9102    0.9029     40435
 weighted avg     0.9114    0.9081    0.9088     40435



In [12]:
# ---------------- Try your own question pairs ----------------
def is_duplicate(q1, q2):
    p = predict([q1], [q2], tta=True)[0]
    return {"probability": round(float(p), 4), "duplicate": bool(p > best_thr)}

print(is_duplicate("How can I learn Python quickly?", "What is the fastest way to learn Python?"))
print(is_duplicate("How can I learn Python quickly?", "What is the capital of France?"))

{'probability': 0.9116, 'duplicate': True}
{'probability': 0.0002, 'duplicate': False}
